# 10: Visualizing the Graph

This notebook demonstrates various ways to visualize the Neo4j knowledge graph, including network diagrams, entity relationship graphs, and interactive visualizations.

## Overview

We'll:
1. Extract graph data from Neo4j
2. Create network visualizations using NetworkX and matplotlib
3. Create interactive visualizations with Plotly
4. Visualize entity relationships and note connections
5. Create subgraph visualizations for specific entities


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter

print("✅ Additional libraries imported")


## Extract Graph Data from Neo4j

Query Neo4j to get nodes and relationships for visualization.


In [ ]:
# Extract entity nodes and relationships
with driver.session(database=settings.neo4j_database) as session:
    # Get entities
    entity_result = session.run("""
        MATCH (e:Entity)
        OPTIONAL MATCH (e)<-[:CONTAINS]-(n:Note)
        RETURN e.name AS name, e.type AS type, 
               count(DISTINCT n) AS note_count
        ORDER BY note_count DESC
        LIMIT 50
    """)
    entities = [record for record in entity_result]
    
    # Get relationships between entities
    rel_result = session.run("""
        MATCH (e1:Entity)-[r]->(e2:Entity)
        RETURN e1.name AS from_entity, 
               type(r) AS rel_type,
               e2.name AS to_entity
        LIMIT 100
    """)
    relationships = [record for record in rel_result]

print(f"✅ Extracted {len(entities)} entities and {len(relationships)} relationships")


## Create NetworkX Graph

Build a NetworkX graph from the extracted data for visualization.


In [ ]:
# Create NetworkX graph
G = nx.DiGraph()

# Add nodes
for entity in entities:
    G.add_node(
        entity["name"],
        entity_type=entity["type"],
        note_count=entity["note_count"]
    )

# Add edges
for rel in relationships:
    G.add_edge(
        rel["from_entity"],
        rel["to_entity"],
        rel_type=rel["rel_type"]
    )

print(f"✅ NetworkX graph created")
print(f"   Nodes: {G.number_of_nodes()}")
print(f"   Edges: {G.number_of_edges()}")


## Static Visualization with Matplotlib

Create a static network diagram using matplotlib.


In [ ]:
# Create a subgraph with top entities (for readability)
top_entities = sorted(entities, key=lambda x: x["note_count"], reverse=True)[:20]
top_entity_names = {e["name"] for e in top_entities}
subgraph = G.subgraph(top_entity_names)

# Create layout
pos = nx.spring_layout(subgraph, k=2, iterations=50)

# Draw the graph
plt.figure(figsize=(16, 12))
nx.draw_networkx_nodes(
    subgraph, pos,
    node_color='lightblue',
    node_size=[subgraph.nodes[n].get('note_count', 1) * 50 for n in subgraph.nodes()],
    alpha=0.7
)
nx.draw_networkx_edges(
    subgraph, pos,
    edge_color='gray',
    alpha=0.5,
    arrows=True,
    arrowsize=10
)
nx.draw_networkx_labels(
    subgraph, pos,
    font_size=8,
    font_weight='bold'
)

plt.title("Entity Relationship Graph (Top 20 Entities)", size=16)
plt.axis('off')
plt.tight_layout()
plt.show()

print("✅ Static visualization created")


## Interactive Visualization with Plotly

Create an interactive network visualization using Plotly.


In [ ]:
# Prepare data for Plotly
edge_x = []
edge_y = []
for edge in subgraph.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines'
)

node_x = []
node_y = []
node_text = []
node_info = []
for node in subgraph.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(node)
    node_info.append(
        f"Entity: {node}<br>"
        f"Type: {subgraph.nodes[node].get('entity_type', 'N/A')}<br>"
        f"Notes: {subgraph.nodes[node].get('note_count', 0)}"
    )

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    hoverinfo='text',
    text=node_text,
    textposition="middle center",
    hovertext=node_info,
    marker=dict(
        showscale=True,
        colorscale='YlGnBu',
        reversescale=True,
        color=[subgraph.nodes[n].get('note_count', 1) for n in subgraph.nodes()],
        size=[subgraph.nodes[n].get('note_count', 1) * 3 + 10 for n in subgraph.nodes()],
        colorbar=dict(
            thickness=15,
            title="Note Count",
            xanchor="left",
            titleside="right"
        ),
        line=dict(width=2)
    )
)

fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title='Interactive Entity Relationship Graph',
        titlefont_size=16,
        showlegend=False,
        hovermode='closest',
        margin=dict(b=20, l=5, r=5, t=40),
        annotations=[dict(
            text="Hover over nodes to see details",
            showarrow=False,
            xref="paper", yref="paper",
            x=0.005, y=-0.002,
            xanchor='left', yanchor='bottom',
            font=dict(color="#888", size=12)
        )],
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
)

fig.show()

print("✅ Interactive visualization created")


In [ ]:
# Count entity types
entity_types = Counter([e["type"] for e in entities])

# Create bar chart
fig = px.bar(
    x=list(entity_types.keys()),
    y=list(entity_types.values()),
    title="Entity Type Distribution",
    labels={"x": "Entity Type", "y": "Count"},
    color=list(entity_types.values()),
    color_continuous_scale="Viridis"
)
fig.update_layout(showlegend=False)
fig.show()

print("✅ Entity type distribution chart created")


## Subgraph Visualization for Specific Entity

Visualize the neighborhood around a specific entity.


In [ ]:
# Select an entity to visualize its neighborhood
entity_name = "Project"  # Change this to an entity from your graph

# Get neighbors of this entity
if entity_name in G:
    neighbors = list(G.neighbors(entity_name)) + list(G.predecessors(entity_name))
    neighbors.append(entity_name)  # Include the entity itself
    
    # Create subgraph
    entity_subgraph = G.subgraph(neighbors)
    
    if entity_subgraph.number_of_nodes() > 0:
        # Create layout
        pos = nx.spring_layout(entity_subgraph, k=1.5, iterations=50)
        
        # Draw
        plt.figure(figsize=(12, 8))
        nx.draw_networkx_nodes(
            entity_subgraph, pos,
            nodelist=[entity_name],
            node_color='red',
            node_size=1000,
            alpha=0.9,
            label=entity_name
        )
        nx.draw_networkx_nodes(
            entity_subgraph, pos,
            nodelist=[n for n in entity_subgraph.nodes() if n != entity_name],
            node_color='lightblue',
            node_size=500,
            alpha=0.7
        )
        nx.draw_networkx_edges(
            entity_subgraph, pos,
            edge_color='gray',
            alpha=0.6,
            arrows=True,
            arrowsize=15
        )
        nx.draw_networkx_labels(
            entity_subgraph, pos,
            font_size=9,
            font_weight='bold'
        )
        
        plt.title(f"Neighborhood Graph for '{entity_name}'", size=14)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        
        print(f"✅ Subgraph visualization created for '{entity_name}'")
        print(f"   Connected to {len(neighbors) - 1} other entities")
    else:
        print(f"No neighbors found for '{entity_name}'")
else:
    print(f"Entity '{entity_name}' not found in graph")
    print("Try a different entity name")


## Relationship Type Analysis

Analyze and visualize relationship types in the graph.


In [ ]:
# Count relationship types
rel_types = Counter([r["rel_type"] for r in relationships])

# Create visualization
fig = px.pie(
    values=list(rel_types.values()),
    names=list(rel_types.keys()),
    title="Relationship Type Distribution"
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

print("✅ Relationship type distribution created")
